In [1]:
import pandas as pd
import geopandas as gpd
import duckdb

In [25]:
#connect to duckdb
con = duckdb.connect('../database/spatial-db.db', read_only=False)

IOException: IO Error: Could not set lock on file "/Users/chuli/Github/mappie-talkie/backend/notebooks/../database/spatial-db.db": Conflicting lock is held in /Users/chuli/opt/anaconda3/envs/spatial-db/bin/python3.10 (PID 63372) by user chuli. See also https://duckdb.org/docs/connect/concurrency

In [4]:
#install the spatial extension
con.install_extension("spatial")
con.load_extension("spatial")

In [5]:
county = gpd.read_file('data/01_county/county_neighbors.shp')

In [6]:
county.head()

,GEOID,ppl_densit,transit_to,walk_to_wo,state_name,county_nam,c_lat,c_lon,rural,county_tot,...,pct_gas,pct_electr,pct_oil,pct_coal,pct_solar,pct_other_,pct_no_fue,main_fuel,neighbors_,geometry
0,31039,15.775040,0.000000,0.017324,Nebraska,Cuming,41.916404,-96.787401,Rural,8846,...,0.646,0.286,0.026,0.020,0.000,0.020,0.002,gas,"['Thurston', 'Dodge', 'Stanton', 'Thurston']","POLYGON ((-96.55516 41.91587, -96.55515 41.914..."
1,53069,17.023660,0.002581,0.041935,Washington,Wahkiakum,46.291134,-123.433470,Rural,4488,...,0.046,0.700,0.027,0.202,0.000,0.005,0.021,electricity,"['Pacific', 'Clark', 'Pacific', 'Cowlitz']","POLYGON ((-123.72755 46.2645, -123.72756 46.26..."
2,35011,0.729626,0.000000,0.056787,New Mexico,De Baca,34.342414,-104.411958,Rural,1748,...,0.375,0.472,0.000,0.057,0.000,0.096,0.000,electricity,"['Guadalupe', 'Chaves', 'Guadalupe', 'Roosevelt']","POLYGON ((-104.89337 34.08894, -104.89337 34.0..."
3,31109,384.524800,0.009055,0.030294,Nebraska,Lancaster,40.784174,-96.687756,Not rural,319090,...,0.640,0.350,0.001,0.003,0.001,0.003,0.002,gas,"['Saunders', 'Gage', 'Seward', 'Cass']","POLYGON ((-96.68493 40.5233, -96.69219 40.5231..."
4,31129,7.114601,0.000000,0.045455,Nebraska,Nuckolls,40.176380,-98.047185,Rural,4148,...,0.636,0.303,0.001,0.022,0.000,0.023,0.015,gas,"['Clay', 'Thayer', 'Webster', 'Thayer']","POLYGON ((-98.2737 40.1184, -98.27374 40.1224,..."


In [7]:
#print all column names
print(county.columns)

Index(['GEOID', 'ppl_densit', 'transit_to', 'walk_to_wo', 'state_name',
       'county_nam', 'c_lat', 'c_lon', 'rural', 'county_tot', 'tot_cov_po',
       'pct_tot_co', 'pct_no_fix', 'no_bb_or_c', 'pct_no_bb_', 'occ_housin',
       'gas', 'electricit', 'oil', 'coal', 'solar', 'other_fuel', 'no_fuel',
       'pct_gas', 'pct_electr', 'pct_oil', 'pct_coal', 'pct_solar',
       'pct_other_', 'pct_no_fue', 'main_fuel', 'neighbors_', 'geometry'],
      dtype='object')


In [10]:
county.drop(columns=['coal', 'solar', 'other_fuel', 'no_fuel', 'pct_coal', 'pct_solar', 'pct_other_', 'pct_no_fue'], inplace=True)

In [12]:
#for columns pct_gas, pct_electr, pct_oil *100
county['pct_gas'] = county['pct_gas'] * 100
county['pct_electr'] = county['pct_electr'] * 100
county['pct_oil'] = county['pct_oil'] * 100

In [13]:
county

,GEOID,ppl_densit,transit_to,walk_to_wo,state_name,county_nam,c_lat,c_lon,rural,county_tot,...,occ_housin,gas,electricit,oil,pct_gas,pct_electr,pct_oil,main_fuel,neighbors_,geometry
0,31039,15.775040,0.000000,0.017324,Nebraska,Cuming,41.916404,-96.787401,Rural,8846,...,3711,2397,1063,95,64.6,28.6,2.6,gas,"['Thurston', 'Dodge', 'Stanton', 'Thurston']","POLYGON ((-96.55516 41.91587, -96.55515 41.914..."
1,53069,17.023660,0.002581,0.041935,Washington,Wahkiakum,46.291134,-123.433470,Rural,4488,...,1954,89,1367,52,4.6,70.0,2.7,electricity,"['Pacific', 'Clark', 'Pacific', 'Cowlitz']","POLYGON ((-123.72755 46.2645, -123.72756 46.26..."
2,35011,0.729626,0.000000,0.056787,New Mexico,De Baca,34.342414,-104.411958,Rural,1748,...,741,278,350,0,37.5,47.2,0.0,electricity,"['Guadalupe', 'Chaves', 'Guadalupe', 'Roosevelt']","POLYGON ((-104.89337 34.08894, -104.89337 34.0..."
3,31109,384.524800,0.009055,0.030294,Nebraska,Lancaster,40.784174,-96.687756,Not rural,319090,...,129869,83119,45452,163,64.0,35.0,0.1,gas,"['Saunders', 'Gage', 'Seward', 'Cass']","POLYGON ((-96.68493 40.5233, -96.69219 40.5231..."
4,31129,7.114601,0.000000,0.045455,Nebraska,Nuckolls,40.176380,-98.047185,Rural,4148,...,1748,1112,530,1,63.6,30.3,0.1,gas,"['Clay', 'Thayer', 'Webster', 'Thayer']","POLYGON ((-98.2737 40.1184, -98.27374 40.1224,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3095,13123,73.950830,0.002572,0.008928,Georgia,Gilmer,34.691178,-84.455627,Rural,31369,...,12868,6518,5644,66,50.7,43.9,0.5,gas,"['Fannin', 'Pickens', 'Murray', 'Fannin']","POLYGON ((-84.30237 34.57832, -84.30329 34.577..."
3096,27135,9.148537,0.001804,0.025258,Minnesota,Roseau,48.775147,-95.810806,Rural,15165,...,5833,3691,1506,194,63.3,25.8,3.3,gas,"['Kittson', 'Marshall', 'Marshall', 'Lake of t...","POLYGON ((-95.25857 48.88666, -95.25707 48.885..."
3097,28089,152.944400,0.000753,0.005686,Mississippi,Madison,32.634710,-90.033714,Not rural,106272,...,42182,23920,18100,17,56.7,42.9,0.0,gas,"['Holmes', 'Rankin', 'Yazoo', 'Leake']","POLYGON ((-90.14883 32.40026, -90.1489 32.4001..."
3098,48227,38.286330,0.000000,0.010094,Texas,Howard,32.306169,-101.435505,Not rural,36664,...,12010,6553,5385,6,54.6,44.8,0.0,gas,"['Borden', 'Glasscock', 'Martin', 'Mitchell']","POLYGON ((-101.18138 32.21252, -101.18138 32.2..."


In [14]:
#save to shapefile
county.to_file('data/01_county/county_neighbors.shp')

In [15]:
#read the shapefile into duckdb
con.sql("SELECT * FROM ST_Read('data/01_county/county_neighbors.shp')")

┌───────┬────────────┬───────────────────┬───────────────────┬───────────────┬─────────────┬────────────────────┬─────────────────────┬───────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────┬────────────┬───────┬────────────────────┬────────────────────┬─────────┬─────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
#drop table if it exists
con.execute("DROP TABLE IF EXISTS county")

In [17]:
#create a table for the state shapefile
con.sql("CREATE TABLE county AS SELECT * FROM ST_Read('data/01_county/county_neighbors.shp')")

In [18]:
#check if the table was created
con.table('county')

┌───────┬────────────┬───────────────────┬───────────────────┬───────────────┬─────────────┬────────────────────┬─────────────────────┬───────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬───────┬────────────┬───────┬────────────────────┬────────────────────┬─────────┬─────────────┬──────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [ ]:
#get all the columns in the table
con.table('county').columns

In [24]:
con.close()